In [ ]:
import pandas as pd
from tqdm import tqdm
import re

import numpy as np
from sentence_transformers import SentenceTransformer
import openai
import pickle
import os

In [ ]:
openai.api_key = 'xxx'
openai_client = openai.OpenAI(api_key=openai.api_key)


In [ ]:
def save_to_pickle(obj, filepath):
    with open(filepath, 'wb') as f:
        pickle.dump(obj, f)

def load_from_pickle(filepath):
    with open(filepath, 'rb') as f:
        return pickle.load(f)

In [ ]:
def preprocess_candidate_terms(df, column_name):
    """
    Returns:
        keyword_map: dict[str, dict]
    """

    series = (
        df[column_name]
        .dropna()
        .str.split(";")
        .explode()
        .str.strip()
    )

    raw_keywords = series.unique()
    keyword_map = {}
    normalized_set = set()

    for raw_kw in tqdm(raw_keywords):
        normalized = raw_kw.lower()
        normalized = re.sub(r"[^a-z0-9\s]", " ", normalized)
        normalized = re.sub(r"\s+", " ", normalized).strip()

        keyword_map[raw_kw] = {
            "normalized": normalized,
            }

        normalized_set.add(normalized)


    return keyword_map

In [ ]:
def normalize_term(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text


def build_processed_keyword_topk_matches(
    processed_keywords,
    mesh_file_path="data/mesh_category.xlsx",
    model_name="all-mpnet-base-v2",
    top_k=10,
):
    mesh_df = pd.read_excel(mesh_file_path)
    mesh_df.columns = [str(c).strip().lower() for c in mesh_df.columns]

    if "term" not in mesh_df.columns or "class" not in mesh_df.columns:
        raise ValueError("mesh_category.xlsx must contain 'term' and 'class' columns")

    mesh_df = mesh_df[["term", "class"]].dropna(subset=["term", "class"]).copy()
    mesh_df["term"] = mesh_df["term"].astype(str).str.strip()
    mesh_df["class"] = mesh_df["class"].astype(str).str.strip()
    mesh_df["normalized_term"] = mesh_df["term"].map(normalize_term)
    mesh_df = mesh_df[mesh_df["normalized_term"] != ""].drop_duplicates(subset=["normalized_term"])

    mesh_terms = mesh_df["normalized_term"].tolist()

    model = SentenceTransformer(model_name)
    processed_embeddings = model.encode(processed_keywords, normalize_embeddings=True, show_progress_bar=True)
    mesh_embeddings = model.encode(mesh_terms, normalize_embeddings=True, show_progress_bar=True)

    # Because embeddings are normalized, dot product == cosine similarity
    similarity_matrix = np.matmul(processed_embeddings, np.array(mesh_embeddings).T)
    top_k = min(top_k, len(mesh_terms))
    top_indices = np.argsort(-similarity_matrix, axis=1)[:, :top_k]

    mesh_lookup = mesh_df.set_index("normalized_term")[["term", "class"]].to_dict("index")

    processed_keyword_topk_matches = {}
    for i, pk in enumerate(processed_keywords):
        matches = []
        for mesh_idx in top_indices[i]:
            matched_norm = mesh_terms[mesh_idx]
            matched_info = mesh_lookup[matched_norm]
            matches.append(
                (
                    matched_info["term"],
                    matched_info["class"],
                    round(float(similarity_matrix[i, mesh_idx]), 3),
                )
            )
        processed_keyword_topk_matches[pk] = matches

    return processed_keyword_topk_matches

In [ ]:
df_raw = pd.read_csv("data/raw/jbi_pubmed_2011_2025_w_keywords.csv")
keyword_map = preprocess_candidate_terms(df_raw, 'keywords')
print("number of keywords:", len(keyword_map))

processed_keywords = []
for kw, kw_info in keyword_map.items():
    processed_keywords.append(kw_info['normalized'])

processed_keywords = list(set(processed_keywords))
print("number of processed keywords:", len(processed_keywords))




processed_keyword_top10_matches = build_processed_keyword_topk_matches(processed_keywords)

# preview one keyword and its top-10 matches
example_key = processed_keywords[0]
example_key, processed_keyword_top10_matches[example_key]

In [ ]:
def generate_gpt4_structured_response(content, print_output=False):
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": content}
        ],
        temperature=0,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "GPTClusterResponse",
                "schema": {
                    "type": "object",
                    "properties": {
                        "M": {"type": "boolean"},
                        "H": {"type": "boolean"},
                    },
                    "required": ["M", "H"],
                    "additionalProperties": False,
                },
            },
        },
    )

    result = response.choices[0].message.content

    if print_output:
        print(result)

    return result


import json

def check_category(keyword: str, similar_keywords, max_attempts: int = 3, verbose=False) -> dict:
    for attempt in range(max_attempts):
        try:
            message = f"""
You are an expert in biomedical informatics taxonomy.

Your task is to classify the following keyword:

Keyword: "{keyword}"

Determine whether the keyword belongs to these categories:
- M: describe a computational method
- H: describe a biomedical domain (e.g., diseases, biological processes, anatomical entities, patient populations)

You are also given the 10 most similar terms with their categories and similarity scores:

{similar_keywords}

Base your decision primarily on semantic meaning. Similar terms should inform but not override your reasoning.

Return your answer strictly as structured boolean fields:
- M: true or false
- H: true or false

Do not provide explanations.
""" 
            if verbose:
                print(message)
            # Call your structured GPT function
            raw_response = generate_gpt4_structured_response(message)
            # Parse JSON safely
            parsed = json.loads(raw_response)

            # Validate structure explicitly (extra safety)
            if isinstance(parsed, dict) and "M" in parsed and "H" in parsed:
                result = {
                    "M": bool(parsed["M"]),
                    "H": bool(parsed["H"]),
                }

                if verbose:
                    print(keyword, "->", result)

                return result

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")

    # Default fallback after max attempts
    return {"M": False, "H": False}


In [ ]:
M_group = []
H_group = []    
for kw in tqdm(processed_keywords):
    categories = check_category(kw, processed_keyword_top10_matches[kw])
    if categories['M']:
        M_group.append(kw)
    if categories['H']:
        H_group.append(kw)
    

print(len(M_group), len(H_group))

os.makedirs("data/processed", exist_ok=True)
save_to_pickle(M_group, "data/processed/M_group.pkl")
save_to_pickle(H_group, "data/processed/H_group.pkl")


In [ ]:
print("number of keywords in the method group: ", len(M_group))
print("number of keywords in the health group: ", len(H_group))
print("number of keywords in both groups: ", len(set(M_group).intersection(set(H_group))))

In [ ]:
from collections import defaultdict
from typing import Dict, Iterable, List


def build_processed_keyword_to_pmids(
    df_raw: pd.DataFrame,
    keyword_map: Dict[str, Dict[str, str]],
    target_keywords: Iterable[str],
    keyword_col: str = "keywords",
    pmid_col: str = "pmid",
) -> Dict[str, List[str]]:
    """
    Build mapping: processed keyword -> list of PMIDs.

    Args:
        df_raw: Raw dataframe containing PMID and keyword columns.
        keyword_map: Mapping from raw keyword to {"normalized": ...}.
        target_keywords: Processed keywords to keep (e.g., M_group or H_group).
        keyword_col: Column in df_raw with ';' separated raw keywords.
        pmid_col: Column in df_raw containing PMID.

    Returns:
        Dict[str, List[str]] where each key is a processed keyword from
        target_keywords and each value is a de-duplicated list of PMIDs.
    """
    if keyword_col not in df_raw.columns:
        raise ValueError(f"Missing column: {keyword_col}")
    if pmid_col not in df_raw.columns:
        raise ValueError(f"Missing column: {pmid_col}")

    target_set = set(target_keywords)
    keyword_to_pmids = defaultdict(list)

    # Prepare all target keys so missing ones still appear with empty lists.
    for kw in target_set:
        keyword_to_pmids[kw]

    for _, row in tqdm(df_raw[[pmid_col, keyword_col]].dropna(subset=[keyword_col]).iterrows()):
        pmid = str(row[pmid_col]).strip()
        raw_keywords = [kw.strip() for kw in str(row[keyword_col]).split(";") if kw.strip()]

        seen_in_row = set()
        for raw_kw in raw_keywords:
            mapped = keyword_map.get(raw_kw)
            processed_kw = mapped.get("normalized") if mapped else normalize_term(raw_kw)

            if processed_kw in target_set and processed_kw not in seen_in_row:
                keyword_to_pmids[processed_kw].append(pmid)
                seen_in_row.add(processed_kw)

    return dict(keyword_to_pmids)




In [ ]:
M_keyword_to_pmids = build_processed_keyword_to_pmids(df_raw, keyword_map, M_group)
H_keyword_to_pmids = build_processed_keyword_to_pmids(df_raw, keyword_map, H_group)
for k in M_keyword_to_pmids:
    if M_keyword_to_pmids[k] == []:
        print(k)

for k in H_keyword_to_pmids:
   if H_keyword_to_pmids[k] == []:
       print(k)

save_to_pickle(M_keyword_to_pmids, "data/processed/M_keyword_to_pmids.pkl")
save_to_pickle(H_keyword_to_pmids, "data/processed/H_keyword_to_pmids.pkl")



In [ ]:
def get_keyword_to_pmid_counts(keyword_to_pmids):
    keyword_to_pmid_counts = defaultdict(int)
    for keyword, pmids in keyword_to_pmids.items():
        keyword_to_pmid_counts[keyword] = len(pmids)
    return keyword_to_pmid_counts

def get_keyword_to_pmid_proportions(keyword_to_pmids):
    keyword_to_pmid_counts = get_keyword_to_pmid_counts(keyword_to_pmids)
    total_pmids = sum(keyword_to_pmid_counts.values())
    keyword_to_pmid_proportions = {keyword: count / total_pmids for keyword, count in keyword_to_pmid_counts.items()}
    return keyword_to_pmid_proportions



In [ ]:
M_keyword_to_pmid_proportions = get_keyword_to_pmid_proportions(M_keyword_to_pmids)
H_keyword_to_pmid_proportions = get_keyword_to_pmid_proportions(H_keyword_to_pmids)


save_to_pickle(M_keyword_to_pmid_proportions, "data/processed/M_keyword_to_pmid_proportions.pkl")
save_to_pickle(H_keyword_to_pmid_proportions, "data/processed/H_keyword_to_pmid_proportions.pkl")



In [ ]:
M_group = set(load_from_pickle("data/processed/M_group.pkl"))
H_group = set(load_from_pickle("data/processed/H_group.pkl"))

